In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv


load_dotenv()

True

In [2]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [3]:
class BlogState(TypedDict):
    title : str
    outline : str
    blog : str
    rating : str

In [4]:
def createOutline(state: BlogState) -> BlogState:
    """ Create an outline based on the provided title"""

    title = state["title"]
    prompt = f"Generate a professional outline for a blog on the topic : {title}"
    outline = model.invoke(prompt).content
    state["outline"] = outline
    return state

In [5]:
def createBlog(state : BlogState) -> BlogState:
    """Create a blog post on the basis of the provided title and outline"""

    title = state["title"]
    outline = state["outline"]
    prompt = f"On the basis of the provided title: {title} and outline: {outline}, generate a blog post."
    blog = model.invoke(prompt).content
    state["blog"] = blog
    print("blog", blog)
    return state



In [6]:
def evaluateBlog(state : BlogState) -> BlogState:
    """Evaluate the blog based the outline"""

    title = state["title"]
    outline = state["outline"]
    blog = state["blog"][0]
    prompt = f"For the topic - {title}, evaluate and give me a rating out of 10 for the attached blog. \n The outlines are : {outline} \n The blog is : {blog}"

    rating = model.invoke(prompt).content
    state["rating"] = rating
    return state


In [7]:
graph = StateGraph(BlogState)
graph.add_node('createOutline', createOutline)
graph.add_node('createBlog', createBlog)
graph.add_node('evaluateBlog', evaluateBlog)

graph.add_edge(START, 'createOutline')
graph.add_edge('createOutline', 'createBlog')
graph.add_edge('createBlog', 'evaluateBlog')
graph.add_edge('evaluateBlog', END)
graph.add_edge('createBlog', END)

workflow = graph.compile()

In [8]:
initial_state = {'title' : 'Industrial Revolution'}
final_state  = workflow.invoke(initial_state)
print(final_state["rating"])


blog [{'type': 'text', 'text': '# The Engine of Change: How the Industrial Revolution Shaped the Modern World\n\nIn the mid-18th century, the pulse of the world was measured by the rhythmic labor of muscles—human and animal. But by the 19th century, that pulse had been replaced by the rhythmic thumping of steam-powered pistons. The Industrial Revolution (c. 1760–1840) was not merely a change in how we made things; it was a fundamental reconfiguration of the human experience. It acted as the ultimate catalyst for unprecedented economic growth, radical social upheaval, and the birth of the modern urban landscape we inhabit today.\n\n## The Spark: Why Britain? Why Then?\n\nThe revolution didn’t happen in a vacuum. It was the result of a perfect storm of conditions that coalesced in Britain:\n\n*   **The Agricultural Revolution:** Improved farming techniques led to a population boom, creating a surplus of labor that was no longer tied to the land.\n*   **Resource Abundance:** Britain sat u